## Example 3: Multi-Product Production and Distribution Problem

This example demonstrates a complex optimization problem involving:
- **Multiple production facilities** (A and B)
- **Multiple products** (represented by index combinations)
- **Transportation** between places
- **Capacity constraints** with efficiency factors

### Problem Description

**Decision Variables:**
- `x[i,j,k,m]`: Production quantity from facility i, destination j, for product type k
  - i ∈ {A, B} (facilities)
  - j ∈ {1, 2, 3, 4, 5} (destination)
  - k ∈ {1, 2, 3} (product type)
  
- `y[i1,i2]`: Transportation flow from facility i1 to destination i2
  - Specifically: y[i1,i2]

**Objective Function:**
Maximize total profit = Revenue - Transportation Costs - Production Costs

- **Revenue Terms:**
  - 1550 × Σ x[i,j,1] (Product type 1)
  - 1600 × Σ x[i,j,2] (Product type 2)
  - 1400 × Σ x[i,j,3] (Product type 3)

- **Transportation Costs:**
  - Destination 1: 10000 + 122 × y[i,j]
  - Destination 2: 12000 + 122 × y[i,j]
  - Destination 3: 80000  + 122 × y[i,j]
  - Destination 4: 70000  + 122 × y[i,j]
  - Destination 5: 90000  + 122 × y[i,j]

- **Production Costs:**
  - Facility A, Product 1: 250 × Σ x[A,j,1]
  - Facility A, Product 2: 320 × Σ x[A,j,2]
  - Facility A, Product 3: 420 × Σ x[A,j,3]
  - Facility B, Product 1: 260 × Σ x[B,j,1]
  - Facility B, Product 2: 210 × Σ x[B,j,2]
  - Facility B, Product 3: 350 × Σ x[B,j,3]

- **Demand for Each Destination(p1, p2, p3)(max):**
  - Destination 1: 400, 250, 100
  - Destination 2: 700, 390, 250
  - Destination 3: 200, 400, 340
  - Destination 4: 800, 400, 400
  - Destination 5: 900, 250, 1130

- **Max Capacity for Each Facility:**
  - Facility A, Product 1: 1200
  - Facility A, Product 2: 1100
  - Facility A, Product 3: 1200
  - Facility B, Product 1: 1400
  - Facility B, Product 2: 1300
  - Facility B, Product 3: 700


In [1]:
from pyomo.environ import *

In [2]:
model = ConcreteModel(name="Multip-Product Production and Distribution")

model.I = Set(initialize=["A", "B"], doc="Production Facilities")
model.J = Set(initialize=[1, 2, 3, 4, 5], doc="Destinations")
model.K = Set(initialize=[1, 2, 3], doc="Product Types")


In [3]:
# Param1
revenue = {1: 1550, 2: 1600, 3: 1400}
model.Revenue = Param(model.K, initialize=revenue, doc="Revenue per unit for each product type")

# Param2
trans_fixed_cost = {1: 10_000, 2: 12_000, 3: 80_000, 4: 70_000, 5: 90_000}
model.TransFixedCost = Param(model.J, initialize=trans_fixed_cost, doc="Fixed transportation cost per destination")

# Param3
model.TransVarCost = Param(initialize=122, doc="Variable transportation cost per unit")

# Param4
prod_cost = {
    ("A", 1): 250, ("A", 2): 320, ("A", 3): 420,
    ("B", 1): 260, ("B", 2): 210, ("B", 3): 350
}
model.ProdCost = Param(model.I, model.K, initialize=prod_cost, doc="Production cost per unit")

# Param5
demand = {
    (1, 1): 400, (1, 2): 250, (1, 3): 100,
    (2, 1): 700, (2, 2): 390, (2, 3): 250,
    (3, 1): 200, (3, 2): 400, (3, 3): 340,
    (4, 1): 800, (4, 2): 400, (4, 3): 400,
    (5, 1): 900, (5, 2): 250, (5, 3): 1130
}
model.Demand = Param(model.J, model.K, initialize=demand, doc='Maximum demand per destination and product')

# Param6
capacity = {
    ('A', 1): 1200, ('A', 2): 1100, ('A', 3): 1200,
    ('B', 1): 1400, ('B', 2): 1300, ('B', 3): 700
}
model.Capacity = Param(model.I, model.K, initialize=capacity, doc='Maximum production capacity')


In [4]:
model.X = Var(model.I, model.J, model.K, domain=NonNegativeReals,
              doc="Production quantity from facility i to destination j for product k")

model.Y = Var(model.I, model.J, domain=NonNegativeReals,
              doc="Total transportation from facility it o destination j")



In [5]:
def objective_rule(model):
    revenue_total = sum(model.Revenue[k] * model.X[i, j, k] 
                        for i in model.I for j in model.J for k in model.K)
    
    trans_cost_total = sum(model.TransFixedCost[j] + model.TransVarCost * model.Y[i, j]
                           for i in model.I for j in model.J)
    
    prod_cost_total = sum(model.ProdCost[i, k] * model.X[i, j, k]
                          for i in model.I for j in model.J for k in model.K)
    
    return revenue_total - trans_cost_total - prod_cost_total

model.profit = Objective(rule=objective_rule, sense=maximize, doc="Maximize total profit")

In [8]:
# Constraint 1
def demand_constraint_rule(model, J, K):
    return sum(model.X[I, J, K] for I in model.I) <= model.Demand[J, K]

model.demand_constraint = Constraint(model.J, model.K, rule=demand_constraint_rule,
                                     doc="Production cannot exceed demand at each destination")


(type=<class 'pyomo.core.base.constraint.IndexedConstraint'>) on block Multip-
Product Production and Distribution with a new Component (type=<class
'pyomo.core.base.constraint.IndexedConstraint'>). This is usually indicative
of a modelling error. To avoid this warning, use block.del_component() and
block.add_component().


In [9]:
# Constraint 2
def capacity_constraint_rule(model, I, K):
    return sum(model.X[I, J, K] for J in model.J) <= model.Capacity[I, K]

model.capacity_constraint = Constraint(model.I, model.K, rule=capacity_constraint_rule,
                                       doc="Production cannot exceed facility capacity")


In [10]:
# Constaint 3
def transportation_flow_rule(model, I, J):
    return model.Y[I, J] == sum(model.X[I, J, K] for K in model.K)

model.transportation_flow_constraint = Constraint(model.I, model.J, rule=transportation_flow_rule,
                                                  doc="Define tranportation flow as sum of the all products")


In [11]:
print("=" * 60)
print("MODEL SUMMARY")
print("=" * 60)
print(f"\nModel Name: {model.name}")
print(f"\nSets:")
print(f"  - Facilities (I): {len(model.I)} elements")
print(f"  - Destinations (J): {len(model.J)} elements")
print(f"  - Product Types (K): {len(model.K)} elements")

print(f"\nDecision Variables:")
print(f"  - x[i,j,k]: {len(model.I) * len(model.J) * len(model.K)} variables")
print(f"  - y[i,j]: {len(model.I) * len(model.J)} variables")
print(f"  - Total variables: {len(model.I) * len(model.J) * len(model.K) + len(model.I) * len(model.J)}")

print(f"\nConstraints:")
print(f"  - Demand constraints: {len(model.J) * len(model.K)}")
print(f"  - Capacity constraints: {len(model.I) * len(model.K)}")
print(f"  - Transportation flow: {len(model.I) * len(model.J)}")
print(f"  - Total constraints: {len(model.J) * len(model.K) + len(model.I) * len(model.K) + len(model.I) * len(model.J)}")

print(f"\nObjective: Maximize Profit")
print("=" * 60)

MODEL SUMMARY

Model Name: Multip-Product Production and Distribution

Sets:
  - Facilities (I): 2 elements
  - Destinations (J): 5 elements
  - Product Types (K): 3 elements

Decision Variables:
  - x[i,j,k]: 30 variables
  - y[i,j]: 10 variables
  - Total variables: 40

Constraints:
  - Demand constraints: 15
  - Capacity constraints: 6
  - Transportation flow: 10
  - Total constraints: 31

Objective: Maximize Profit


In [13]:
solver = SolverFactory("glpk")

results = solver.solve(model, tee=True)


GLPSOL--GLPK LP/MIP Solver 5.0
Parameter(s) specified in the command line:
 --write /var/folders/ct/s0pvhhv576s4pblf608mg7bc0000gn/T/tmpcc_2u9bh.glpk.raw
 --wglp /var/folders/ct/s0pvhhv576s4pblf608mg7bc0000gn/T/tmpppzl8eo_.glpk.glp
 --cpxlp /var/folders/ct/s0pvhhv576s4pblf608mg7bc0000gn/T/tmpush_1h3b.pyomo.lp
Reading problem data from '/var/folders/ct/s0pvhhv576s4pblf608mg7bc0000gn/T/tmpush_1h3b.pyomo.lp'...
31 rows, 41 columns, 100 non-zeros
284 lines were read
Writing problem data to '/var/folders/ct/s0pvhhv576s4pblf608mg7bc0000gn/T/tmpppzl8eo_.glpk.glp'...
238 lines were written
GLPK Simplex Optimizer 5.0
31 rows, 41 columns, 100 non-zeros
Preprocessing...
21 rows, 30 columns, 60 non-zeros
Scaling...
 A: min|aij| =  1.000e+00  max|aij| =  1.000e+00  ratio =  1.000e+00
Problem data seem to be well scaled
Constructing initial basis...
Size of triangular part is 21
*     0: obj =  -5.240000000e+05 inf =   0.000e+00 (30)
*    18: obj =   6.304020000e+06 inf =   0.000e+00 (0)
OPTIMAL LP 

In [15]:
if results.solver.termination_condition == TerminationCondition.optimal:
    print("\n" + "=" * 60)
    print("OPTIMAL SOLUTION FOUND!")
    print("=" * 60)
    
    # Display optimal profit
    print(f"\n💰 Maximum Profit: ${value(model.profit):,.2f}")
    print("=" * 60)
    
    # Display production plan (only non-zero values)
    print("\n📦 PRODUCTION PLAN (x[i,j,k]):")
    print("-" * 60)
    print(f"{'Facility':<10} {'Dest':<6} {'Product':<8} {'Quantity':<10}")
    print("-" * 60)
    
    for i in model.I:
        for j in model.J:
            for k in model.K:
                qty = value(model.X[i, j, k])
                if qty > 0.01:  # Only show non-zero values
                    print(f"{i:<10} {j:<6} {k:<8} {qty:>10.2f}")
    
    print("-" * 60)
    
    # Display transportation flows
    print("\n🚚 TRANSPORTATION FLOWS (y[i,j]):")
    print("-" * 60)
    print(f"{'From Facility':<15} {'To Destination':<18} {'Total Units':<12}")
    print("-" * 60)
    
    for i in model.I:
        for j in model.J:
            flow = value(model.Y[i, j])
            if flow > 0.01:  # Only show non-zero values
                print(f"{i:<15} {j:<18} {flow:>12.2f}")
    
    print("-" * 60)
    
else:
    print("\n⚠️ WARNING: Optimal solution not found!")
    print(f"Termination condition: {results.solver.termination_condition}")


OPTIMAL SOLUTION FOUND!

💰 Maximum Profit: $6,304,020.00

📦 PRODUCTION PLAN (x[i,j,k]):
------------------------------------------------------------
Facility   Dest   Product  Quantity  
------------------------------------------------------------
A          1      1            400.00
A          2      1            700.00
A          3      1            100.00
A          4      2            140.00
A          4      3            390.00
A          5      2            250.00
A          5      3            810.00
B          1      2            250.00
B          1      3            100.00
B          2      2            390.00
B          2      3            250.00
B          3      1            100.00
B          3      2            400.00
B          3      3            340.00
B          4      1            800.00
B          4      2            260.00
B          4      3             10.00
B          5      1            500.00
------------------------------------------------------------

🚚 TRA

In [17]:
if results.solver.termination_condition == TerminationCondition.optimal:
    print("\n" + "=" * 60)
    print("PRODUCTION ANALYSIS BY FACILITY")
    print("=" * 60)
    
    for i in model.I:
        print(f"\n🏭 Facility {i}:")
        print("-" * 40)
        for k in model.K:
            total_prod = sum(value(model.X[i, j, k]) for j in model.J)
            capacity_used = (total_prod / model.Capacity[i, k]) * 100
            print(f"  Product {k}:")
            print(f"    Production: {total_prod:>8.2f} units")
            print(f"    Capacity:   {model.Capacity[i, k]:>8.0f} units")
            print(f"    Utilization: {capacity_used:>6.1f}%")
            print()


PRODUCTION ANALYSIS BY FACILITY

🏭 Facility A:
----------------------------------------
  Product 1:
    Production:  1200.00 units
    Capacity:       1200 units
    Utilization:  100.0%

  Product 2:
    Production:   390.00 units
    Capacity:       1100 units
    Utilization:   35.5%

  Product 3:
    Production:  1200.00 units
    Capacity:       1200 units
    Utilization:  100.0%


🏭 Facility B:
----------------------------------------
  Product 1:
    Production:  1400.00 units
    Capacity:       1400 units
    Utilization:  100.0%

  Product 2:
    Production:  1300.00 units
    Capacity:       1300 units
    Utilization:  100.0%

  Product 3:
    Production:   700.00 units
    Capacity:        700 units
    Utilization:  100.0%



In [19]:
if results.solver.termination_condition == TerminationCondition.optimal:
    print("\n" + "=" * 60)
    print("DEMAND SATISFACTION ANALYSIS")
    print("=" * 60)
    
    for j in model.J:
        print(f"\n📍 Destination {j}:")
        print("-" * 40)
        for k in model.K:
            total_supplied = sum(value(model.X[i, j, k]) for i in model.I)
            demand_satisfaction = (total_supplied / model.Demand[j, k]) * 100
            print(f"  Product {k}:")
            print(f"    Supplied: {total_supplied:>8.2f} units")
            print(f"    Demand:   {model.Demand[j, k]:>8.0f} units")
            print(f"    Satisfied: {demand_satisfaction:>6.1f}%")
            print()


DEMAND SATISFACTION ANALYSIS

📍 Destination 1:
----------------------------------------
  Product 1:
    Supplied:   400.00 units
    Demand:        400 units
    Satisfied:  100.0%

  Product 2:
    Supplied:   250.00 units
    Demand:        250 units
    Satisfied:  100.0%

  Product 3:
    Supplied:   100.00 units
    Demand:        100 units
    Satisfied:  100.0%


📍 Destination 2:
----------------------------------------
  Product 1:
    Supplied:   700.00 units
    Demand:        700 units
    Satisfied:  100.0%

  Product 2:
    Supplied:   390.00 units
    Demand:        390 units
    Satisfied:  100.0%

  Product 3:
    Supplied:   250.00 units
    Demand:        250 units
    Satisfied:  100.0%


📍 Destination 3:
----------------------------------------
  Product 1:
    Supplied:   200.00 units
    Demand:        200 units
    Satisfied:  100.0%

  Product 2:
    Supplied:   400.00 units
    Demand:        400 units
    Satisfied:  100.0%

  Product 3:
    Supplied:   340.

In [21]:
if results.solver.termination_condition == TerminationCondition.optimal:
    print("\n" + "=" * 60)
    print("FINANCIAL BREAKDOWN")
    print("=" * 60)
    
    # Calculate total revenue
    total_revenue = sum(model.Revenue[k] * value(model.X[i, j, k]) 
                       for i in model.I for j in model.J for k in model.K)
    
    # Calculate transportation costs
    total_trans_cost = sum(model.TransFixedCost[j] + model.TransVarCost * value(model.Y[i, j])
                          for i in model.I for j in model.J if value(model.Y[i, j]) > 0.01)
    
    # Calculate production costs
    total_prod_cost = sum(model.ProdCost[i, k] * value(model.X[i, j, k])
                         for i in model.I for j in model.J for k in model.K)
    
    print(f"\n💵 Total Revenue:           ${total_revenue:>12,.2f}")
    print(f"📦 Production Costs:        ${total_prod_cost:>12,.2f}")
    print(f"🚚 Transportation Costs:    ${total_trans_cost:>12,.2f}")
    print("-" * 60)
    print(f"💰 NET PROFIT:              ${value(model.profit):>12,.2f}")
    print("=" * 60)
    
    # Profit margin
    profit_margin = (value(model.profit) / total_revenue) * 100
    print(f"\n📊 Profit Margin: {profit_margin:.2f}%")


FINANCIAL BREAKDOWN

💵 Total Revenue:           $9,394,000.00
📦 Production Costs:        $1,810,800.00
🚚 Transportation Costs:    $1,279,180.00
------------------------------------------------------------
💰 NET PROFIT:              $6,304,020.00

📊 Profit Margin: 67.11%
